# 01 - Train the APS Failure Model

This notebook trains the model used by the FastAPI service. I load the original Scania APS data, check the class imbalance, train an XGBoost pipeline, evaluate it on the held-out test set, and save the final model bundle for inference.


## Imports and paths

Keep the notebook paths explicit so it is clear which files are source data and which file becomes the production model artifact.


In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

TRAIN_PATH = Path("../data/aps_failure_training_set.csv")
TEST_PATH = Path("../data/aps_failure_test_set.csv")
MODEL_PATH = Path("../model.joblib")


## Load the training data

The original CSV has 20 metadata rows before the header. Missing sensor values are stored as `na`, so I load those directly as missing values.


In [ ]:
train_df = pd.read_csv(TRAIN_PATH, skiprows=20, na_values=["na"])
train_df.head()


## Inspect missing values

APS sensor data has many sparse columns. The model pipeline will handle this with mean imputation, but I still check the missingness before training.


In [ ]:
missing_counts = train_df.isna().sum().sort_values(ascending=False)
missing_counts.head(10)


## Encode the target

The dataset labels are `neg` and `pos`. I map them to 0 and 1 so the classifier can learn a binary target.


In [ ]:
train_df["class"] = train_df["class"].map({"neg": 0, "pos": 1})
train_df["class"].value_counts(normalize=True)


## Split features and target

The positive class is rare, so I keep the class counts visible before setting the XGBoost imbalance weight.


In [ ]:
X_train = train_df.drop(columns=["class"])
y_train = train_df["class"]

y_train.value_counts()


## Class imbalance weight

XGBoost uses `scale_pos_weight` to make positive APS failures count more during training.


In [ ]:
num_neg = (y_train == 0).sum()
num_pos = (y_train == 1).sum()
scale_pos_weight = num_neg / num_pos
scale_pos_weight


## Build the pipeline

The saved artifact should include preprocessing and the model together, so the API uses exactly the same imputation, scaling, and feature order as training.


In [ ]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
)

xgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", xgb_model),
])


## Cross-validation

I use F1 rather than accuracy because the positive class is small and accuracy would mostly reward predicting the majority class.


In [ ]:
cv_f1 = cross_val_score(
    xgb_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="f1",
)

cv_f1.mean(), cv_f1


## Train the final model

After the cross-validation check, I fit the pipeline on the full training set before testing on the separate Scania test file.


In [ ]:
xgb_pipeline.fit(X_train, y_train)


## Load the held-out test set

The test CSV follows the same format as the training CSV. I apply the same target encoding before evaluation.


In [ ]:
test_df = pd.read_csv(TEST_PATH, skiprows=20, na_values=["na"])
test_df["class"] = test_df["class"].map({"neg": 0, "pos": 1})

X_test = test_df.drop(columns=["class"])
y_test = test_df["class"]


## Evaluate

I report accuracy, F1, recall, and ROC-AUC. Recall matters here because missing a real APS failure is much more expensive than sending a truck for an unnecessary check.


In [ ]:
y_pred = xgb_pipeline.predict(X_test)
y_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


## Save the model bundle

The API needs both the fitted pipeline and the original feature order. Saving them together avoids feature-order mistakes during inference.


In [ ]:
feature_names = X_train.columns.tolist()

bundle = {
    "model": xgb_pipeline,
    "feature_names": feature_names,
}

joblib.dump(bundle, MODEL_PATH)
